# PART -1

### part - 1.1

In [ ]:
import csv
import json

def parse_row(row):
    converted = dict(row)
    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])
    return converted

accepted = []
rejected = []


with open("appointments.csv",'r',newline= "",encoding = "utf-8") as f:
    r = csv.DictReader(f)
    print("Header:", r.fieldnames)

    row_count = 0
    for row in r:
        row_count += 1
        try:
            parsed = parse_row(row)
            accepted.append(parsed)
        except ValueError:
            rejected.append(row)

    # Print the total row count
    print("Total Row Count:", row_count)

# Save rejected rows to a JSON file
with open("rejected_rows.json", mode="w", encoding="utf-8") as json_file:
    json.dump(rejected, json_file, indent=2)


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Row Count: 1020


### Task 1.2 — Inspect and clean, pass 1

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments.csv")

# shape of data frame created by the
print("shape of the data frame generated is",df.shape)

print("\n")
print("***"*19)
print("\n")

#information of the file
print( df.info())



shape of the data frame generated is (1020, 14)


*********************************************************


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory

In [ ]:
# null data points
print("number of null data points in each column",df.isnull().sum())
print("\n")
# the number of duplicates
print("the number of complete duplicates are ",df.duplicated().sum())

number of null data points in each column appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64


the number of complete duplicates are  20


In [ ]:
# (a) dropping duplicates
df = df.drop_duplicates().copy()
print("row count after removing duplicates:",df.shape[0])
print("\n")

# (b) clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("row count after cleaning booking_channel:",df.shape[0])
print("\n")

# (c) convert appt_date and  booked_on
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"], errors ="coerce")
print("row count after correcting appt_dates and booked_on :",df.shape[0])
print("\n")

# (d) (d) drop rows where appt_date is missing
df = df.dropna(subset=["appt_date","status"])
print("row count after dropping rows where appt_date is missing :",df.shape[0])
print("\n")




row count after removing duplicates: 1000


row count after cleaning booking_channel: 1000


row count after correcting appt_dates and booked_on : 1000


row count after dropping rows where appt_date is missing : 990




### Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
# (a) cleaning age

df["age"] = pd.to_numeric(df["age"], errors = "coerce")
df.loc[(df["age"]<1)|(df["age"]>100), "age"] = np.nan

# (b) managing and filling the missing values

df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"]= df["distance_km"].fillna(df["distance_km"].median())


# (c) For distance_km and fee_inr, compute Q1, Q3 and IQR

# for distance

q1 = df["distance_km"].quantile(0.25)
q3 = df["distance_km"].quantile(0.75)

iqr = q3-q1

lwr = q1 - 1.5*iqr
upr = q3 + 1.5*iqr

lwr_cpd = (df["distance_km"] < lwr).sum()
upr_cpd = (df["distance_km"] < upr).sum()
print(f"the number of values capped in the distance column is {lwr_cpd + upr_cpd}")

df["distance_km"] = np.clip(df["distance_km"],lwr,upr)

# for fees

Q1 = df["fee_inr"].quantile(0.25)
Q3 = df["fee_inr"].quantile(0.75)

Iqr = q3-q1

Lwr = q1 - 1.5*iqr
Upr = q3 + 1.5*iqr

Lwr_cpd = (df["fee_inr"]  < Lwr).sum()
Upr_cpd = (df["fee_inr"] < Upr).sum()
print(f"the number of values capped in the fees column is {Lwr_cpd + Upr_cpd}")

df["fee_inr"] = np.clip(df["fee_inr"],Lwr,Upr)

#(d) Rounding off values of distance ad fees
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)




the number of values capped in the distance column is 964
the number of values capped in the fees column is 0


### Task 1.4 — Features and a NumPy summary

In [ ]:
# days between bookings and appointment date
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# showed up on appointments
df["no_show"] = (df["status"] == "No-Show").astype(int)

X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy()

#(a) the mean of each column in ["age","distance_km","fee_inr","lead_days"]

mn = np.mean(X, axis = 0)
print(f'the mean of the columns in ["age","distance_km","fee_inr","lead_days"] are {mn}')
print("\n")
print("-*-"*33)

#(b)  the number and share of rows with leading days > 14
mask = df["lead_days"] > 14
ld_rws = mask.sum()
shr = mask.mean()

print("\n")
print(f"the number and share of rows with leading days > 14 are {ld_rws} and {shr} respectively")

#(c) z-scores of the distance_km
dist = X[:1]
z_scrs = ( dist - np.mean(dist)) / np.std(dist)
outlrs = (np.abs(z_scrs)>2).sum()

print("\n")
print(f" the z-scores of distance and the number of rows having |z|>2 are {z_scrs} and {outlrs}")



the mean of the columns in ["age","distance_km","fee_inr","lead_days"] are [48.17373737  9.50090909 26.         11.81919192]


-*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*--*-


the number and share of rows with leading days > 14 are 296 and 0.298989898989899 respectively


 the z-scores of distance and the number of rows having |z|>2 are [[ 1.57157464 -0.38416269 -0.01164129 -1.17577066]] and 0


In [ ]:
# summary

nume_cols = ["age","distance_km","fee_inr","lead_days"]
clm_mean = np.array([df[col].mean() for col in nume_cols])

print("column means :",np.round(clm_mean,2))

# lead day stats

min_ld = df["lead_days"].min()
max_ld = df["lead_days"].max()

pct_lead_gt_14 = (df["lead_days"]>14).mean()*100
print(f"Lead Days range: {min_ld} to {max_ld}")
print(f"Rows with lead days greayter than 14: {pct_lead_gt_14:.1f}%")

# no-shows count and percentage
no_shws = (df["status"] == "No-Show").sum()
tot_rows = len(df)
print(f"No-shows: {no_shws} out of {tot_rows} ({no_shws/tot_rows*100:.1f}%)")

# Distance Z-scores (|z| > 2) check
dist_mn = df["distance_km"].mean()
dist_std = df["distance_km"].std(ddof = 0)
z_scrs = np.abs((df["distance_km"]-dist_mn)/dist_std)
outliers_z2 = (z_scrs > 2).sum()
print(f"rows with |Z| > 2 for distance_km: {outliers_z2}")

column means : [48.17  9.5  26.   11.82]
Lead Days range: 1 to 45
Rows with lead days greayter than 14: 29.9%
No-shows: 218 out of 990 (22.0%)
rows with |Z| > 2 for distance_km: 53


### Task 1.5 — Enrich the clinics from a live API

In [ ]:
import requests
import json
import csv
import pandas as pd
import numpy as np

# opening the clincs file containing the info about the different clinics across 6 cities
with open("clinics.csv",'r',newline= "",encoding = "utf-8") as f:
    r = csv.DictReader(f)
    print("Header:", r.fieldnames)

cl = pd.read_csv("clinics.csv")

print("\n")
print("-----"*32)
# checking for any missing values
print("number of null data points in each column",cl.isnull().sum())

temps = []
for _,row in cl.iterrows():
    cln_name = row["clinic_name"]
    params = {
        "latitude" : row["latitude"],
        "longitude" : row["longitude"],
        "hourly" : "temperature_2m",
        "forecast_days" : 1
    }
    temp_c = None
    failure_reason = ""

    # attempting the request upto 2 times
    for attempt in range(2):
        try:
            res = reply =requests.get("https://api.open-meteo.com/v1/forecast", params = params,timeout = 10)

            if res.status_code == 200:
                data = res.json()

                if "hourly" in data and "temperature_2m" in data["hourly"] and len(data["hourly"]["temperature_2m"]) >0 :
                    temp_c = data["hourly"]["temperature_2m"][0]
                    failure_reason = ""
                    break

                else :
                    failure_reason = "Missing 'hourly' or 'temperature_2m' data in response "

            else:
                failure_reason = f"HTTP status code {res.status_code}"

        except requests.exceptions.RequestException as e:
            failure_reason = f"RequestException: {e}"

    # if the temp_c is None even after both the attempts
    if temp_c is None :
        print(f"Failed for {cln_name}: {failure_reason}")

    temps.append(temp_c)

#add the new column and save without the dataframe index
cl["temp_c"] = temps
cl.to_csv("clinics_enriched.csv", index = False)

Header: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


----------------------------------------------------------------------------------------------------------------------------------------------------------------
number of null data points in each column clinic_id      0
clinic_name    0
city           0
tier           0
latitude       0
longitude      0
dtype: int64


In [ ]:
cl

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,19.4
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,19.7
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,19.7
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,16.4
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,16.4
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,26.3
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,16.1
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,14.7


### Task 1.6 — Export the clean table and write the SQL load script

In [ ]:

df = df.sort_values(by="appointment_id").reset_index(drop=True)
df["booked_on"] = pd.to_datetime(df["booked_on"]).dt.strftime("%y-%m-%d")
df["appt_date"] = pd.to_datetime(df["appt_date"]).dt.strftime("%y-%m-%d")

cols =["appointment_id", "patient_id", "clinic_id","booked_on", "appt_date", "age", "gender", "distance_km", "fee_inr", "booking_channel","visit_type", "reminder_sent", "prior_no_shows", "status","lead_days","no_show"]

df_cln = df[cols]

is_uq = df_cln["appointment_id"].is_unique
no_null = df_cln.isna().sum().sum()

print("appointment id that are unique:",is_uq)
print("No null values in any of the 16 columns:",no_null)

assert is_uq, "Error : Duplicate appointment ids found!"
assert no_null == 0,"Error: Missing values detected!"

df_cln.to_csv("appointments_clean.csv", index = False)

clinics_df = pd.read_csv("clinics_enriched.csv")
def format_sql_value(val):
    if isinstance(val,str):
        esc = val.replace("'","''")
        return f"'{esc}'"
    return f"{val}"

with open("load_data.sql","w",encoding = "utf-8") as f :
    for _,row in clinics_df.iterrows():
        values_str = ",".join(format_sql_value(v) for v in row)
        f.write(f"INSERT INTO clinics VALUES ({values_str});\n")
    for _,row in df_cln.iterrows():
        stmt = (
            f"INSERT INTO appointments VALUES("
            f"'{row['appointment_id']}','{row['patient_id']}','{row['clinic_id']}',"
            f"'{row['booked_on']}','{row['appt_date']}',{row['age']},"
            f"'{row['gender']}',{row['distance_km']},{row['fee_inr']},"
            f"'{row['booking_channel']}','{row['visit_type']}',"
            f"{format_sql_value(row['reminder_sent'])},{row['prior_no_shows']},"
            f"'{row['status']}',{row['lead_days']},{row['no_show']});\n"
        )
        f.write(stmt)


appointment id that are unique: True
No null values in any of the 16 columns: 0


In [ ]:
import os
print(os.getcwd())
print(os.path.exists("appointments_clean.csv"))

/content
True


In [ ]:
print(len(cols))


16


In [ ]:
check_csv =pd.read_csv("appointments_clean.csv")
print("CSV shape:",check_csv.shape)
print(check_csv.head(2))

with open("load_data.sql","r",encoding = "utf-8") as f:
    lines = f.readlines()

print("\nTotal SQL statements:",len(lines))
print("First line (clinics):",lines[0].strip())
print("Last line (appointments):", lines[-1].strip())

CSV shape: (990, 16)
  appointment_id patient_id clinic_id booked_on appt_date  age gender  \
0          A0001       P509       C05  25-01-24  25-01-25   60      F   
1          A0002       P046       C07  25-05-27  25-06-06   39      M   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0       26           Phone  Consultation           Yes   
1         12.5       26         Walk-In     Follow-up            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show          1        1  
1               2  Attended         10        0  

Total SQL statements: 998
First line (clinics): INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,19.4);
Last line (appointments): INSERT INTO appointments VALUES('A1000','P085','C08','24-12-03','25-01-17',38,'M',10.8,26,'App','Follow-up','Yes',2,'No-Show',45,1);


In [ ]:
df.isnull().sum()

,0
appointment_id,0
patient_id,0
clinic_id,0
booked_on,0
appt_date,0
age,0
gender,0
distance_km,0
fee_inr,0
booking_channel,0


# PART 2

### PART - 2.2

In [ ]:
import pandas as pd

# Merge your cleaned datasets on clinic_id
merged_df = pd.merge(df_cln, clinics_df, on='clinic_id', how='inner')

# Group by clinic name and city to match the SQL query
python_clinic_summary = merged_df.groupby(['clinic_name', 'city']).agg(
    appointments=('appointment_id', 'count'),
    no_shows=('no_show', 'sum')
).reset_index()

# Calculate percentage and filter for at least 95 appointments
python_clinic_summary['no_show_pct'] = round(100.0 * python_clinic_summary['no_shows'] / python_clinic_summary['appointments'], 1)
python_filtered = python_clinic_summary[python_clinic_summary['appointments'] >= 95]

# Sort by no_show_pct descending to align with SQL order
python_filtered = python_filtered.sort_values(by='no_show_pct', ascending=False)

print("Python Clinic Summary Verification:")
print(python_filtered)
print("Without HAVING, two more clinics appear: Gomti Nagar Care Point (Lucknow) has the highest rate, 33.3%, but only 90 appointments; Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%.\nThe pandas counts match the SQL counts exactly.")

Python Clinic Summary Verification:
                clinic_name    city  appointments  no_shows  no_show_pct
2      Hadapsar Care Centre    Pune           143        39         27.3
4      Malviya Nagar Clinic  Jaipur           142        38         26.8
3     Kothrud Family Clinic    Pune           160        36         22.5
7        Vijay Nagar Clinic  Indore           101        21         20.8
0  Edappally Medical Centre   Kochi           142        23         16.2
6     Vaishali Nagar Clinic  Jaipur           130        18         13.8
Without HAVING, two more clinics appear: Gomti Nagar Care Point (Lucknow) has the highest rate, 33.3%, but only 90 appointments; Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%.
The pandas counts match the SQL counts exactly.


In [ ]:
is_uq = df_cln['appointment_id'].is_unique
no_null = df_cln.isnull().sum().sum()

assert is_uq, "Error: Duplicate appointment IDs found!"
assert no_null ==0, "Error: Missing values found!"

print("Appointment ID that are unique:",is_uq)
print("No null values in any of the 16 columns:", no_null)

df_cln = df_cln.sort_values(by="appointment_id").reset_index(drop = True)
df_cln["booked_on"] = pd.to_datetime(df_cln["booked_on"]).dt.strftime("%y-%m-%d")
df_cln["appt_date"] = pd.to_datetime(df_cln["appt_date"]).dt.strftime("%y-%m-%d")

Appointment ID that are unique: True
No null values in any of the 16 columns: 0


/tmp/ipykernel_687/4136596732.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cln["booked_on"] = pd.to_datetime(df_cln["booked_on"]).dt.strftime("%y-%m-%d")
/tmp/ipykernel_687/4136596732.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cln["appt_date"] = pd.to_datetime(df_cln["appt_date"]).dt.strftime("%y-%m-%d")


### COMPARING PYTHON AND SQL RESULTS AND SUMMARIZATION

In [ ]:
print("please upload the csv files to colab or the server youu are using, like part - 2.2, 2.3 etc to server before runninh all the cells")

please upload the csv files to colab or the server youu are using, like part - 2.2, 2.3 etc to server before runninh all the cells


In [ ]:
import pandas as pd


# Load clean base tables
df_cln = pd.read_csv("appointments_clean.csv")
clinics_df = pd.read_csv("clinics_enriched.csv")

# Ensure dates are parsed cleanly without UserWarning
df_cln["appt_date_dt"] = pd.to_datetime(df_cln["appt_date"], format="%Y-%m-%d", errors="coerce")

# ==========================================
# Task 2.2 — No-Show Rate by Clinic (>= 95 appts)
# ==========================================
print("=== TASK 2.2 COMPARISON ===")
merged_df = pd.merge(df_cln, clinics_df, on="clinic_id", how="inner")

py_2_2 = (
    merged_df.groupby(["clinic_name", "city"])
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
)
py_2_2["no_show_pct"] = (100.0 * py_2_2["no_shows"] / py_2_2["appointments"]).round(1)
py_2_2 = (
    py_2_2[py_2_2["appointments"] >= 95]
    .sort_values(by="no_show_pct", ascending=False)
    .reset_index(drop=True)
)

sql_2_2 = pd.read_csv("part_2.2_no-show-rate.csv").reset_index(drop=True)
print("Row counts match:", len(py_2_2), "vs", len(sql_2_2))
print(py_2_2.compare(sql_2_2, result_names=("python", "sql")))


# ==========================================
# Task 2.3 — Lead-Time Bands
# ==========================================
print("\n=== TASK 2.3 COMPARISON ===")
def lead_band(days):
    if 1 <= days <= 7:
        return "1-7"
    elif 8 <= days <= 14:
        return "8-14"
    elif 15 <= days <= 30:
        return "15-30"
    else:
        return "31+"

band_order = {"1-7": 1, "8-14": 2, "15-30": 3, "31+": 4}
df_2_3 = df_cln.copy()
df_2_3["lead_band"] = df_2_3["lead_days"].apply(lead_band)

py_2_3 = (
    df_2_3.groupby("lead_band")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
)
py_2_3["no_show_pct"] = ((py_2_3["no_shows"] / py_2_3["appointments"]) * 100).round(2)
py_2_3["sort_key"] = py_2_3["lead_band"].map(band_order)
py_2_3 = py_2_3.sort_values("sort_key").drop(columns="sort_key").reset_index(drop=True)

sql_2_3 = pd.read_csv("part-2.3-booking-far-ahead.csv").reset_index(drop=True)
print("Row counts match:", len(py_2_3), "vs", len(sql_2_3))
print(py_2_3.compare(sql_2_3, result_names=("python", "sql")))


# ==========================================
# Task 2.4 — Month-by-Month Trend & Prior-Month Diff
# ==========================================
print("\n=== TASK 2.4 COMPARISON ===")
df_2_4 = df_cln.copy()
df_2_4["appt_month"] = df_2_4["appt_date"].astype(str).str.slice(0, 7)

py_2_4 = (
    df_2_4.groupby("appt_month")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
    .sort_values("appt_month")
    .reset_index(drop=True)
)
py_2_4["no_show_pct"] = ((py_2_4["no_shows"] / py_2_4["appointments"]) * 100).round(2)
py_2_4["rate_diff_vs_prior_month"] = py_2_4["no_show_pct"].diff().round(2)
py_2_4 = py_2_4[["appt_month", "appointments", "no_show_pct", "rate_diff_vs_prior_month"]]

sql_2_4 = pd.read_csv("part-2.4-monthly-trend-and-stats.csv").reset_index(drop=True)

# Inspect exact columns and shapes for Part 2.4
print("py_2_4 columns:", list(py_2_4.columns), "shape:", py_2_4.shape)
print("sql_2_4 columns:", list(sql_2_4.columns), "shape:", sql_2_4.shape)

# Align columns to match the Part 2.4 SQL output
cols_2_4 = ["appt_month", "appointments", "no_show_pct", "rate_diff_vs_prior_month"]

py_2_4_cmp = py_2_4[cols_2_4].reset_index(drop=True)
sql_2_4_cmp = sql_2_4[cols_2_4].reset_index(drop=True)

# Round floats identically before comparing
for col in ["no_show_pct", "rate_diff_vs_prior_month"]:
    py_2_4_cmp[col] = py_2_4_cmp[col].round(2)
    sql_2_4_cmp[col] = sql_2_4_cmp[col].round(2)

# Run compare if shapes match
if py_2_4_cmp.shape == sql_2_4_cmp.shape:
    diff_2_4 = py_2_4_cmp.compare(sql_2_4_cmp, result_names=("python", "sql"))
    print(diff_2_4 if not diff_2_4.empty else "Part 2.4 matches with zero differences!")
else:
    print("Shape mismatch in Part 2.4 — check month filtering or grouping.")

print("Row counts match:", len(py_2_4), "vs", len(sql_2_4))
print("py_2_4 cols:", list(py_2_4.columns), "shape:", py_2_4.shape)
print("sql_2_4 cols:", list(sql_2_4.columns), "shape:", sql_2_4.shape)

# Keep only shared columns in the exact same order and reset index
common_cols_2_4 = [c for c in sql_2_4.columns if c in py_2_4.columns]
py_2_4_aligned = py_2_4[common_cols_2_4].reset_index(drop=True)
sql_2_4_aligned = sql_2_4[common_cols_2_4].reset_index(drop=True)

# Round numeric rate columns so tiny float differences don't trigger false mismatches
for col in ["no_show_pct", "rate_diff_vs_prior_month"]:
    if col in py_2_4_aligned.columns:
        py_2_4_aligned[col] = pd.to_numeric(py_2_4_aligned[col], errors="coerce").round(2)
        sql_2_4_aligned[col] = pd.to_numeric(sql_2_4_aligned[col], errors="coerce").round(2)

if py_2_4_aligned.shape == sql_2_4_aligned.shape:
    diff_2_4 = py_2_4_aligned.compare(sql_2_4_aligned, result_names=("python", "sql"))
    print(diff_2_4 if not diff_2_4.empty else "Part 2.4 matches with zero differences!")
else:
    print("Shape mismatch — check month rows or filtering in py_2_4 vs sql_2_4.")


# ==========================================
# Task 2.5 — Clinic Ranks Inside Each City & vs Overall
# ==========================================
print("\n=== TASK 2.5 COMPARISON ===")
overall_rate = (df_cln["no_show"].sum() * 100.0) / len(df_cln)

py_2_5 = (
    merged_df.groupby(["clinic_name", "city"])
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
)
py_2_5["no_show_pct"] = ((py_2_5["no_shows"] / py_2_5["appointments"]) * 100).round(2)
py_2_5["rank_in_city"] = (
    py_2_5.groupby("city")["no_show_pct"]
    .rank(ascending=False, method="min")
    .astype(int)
)

def vs_overall_label(pct):
    if pct > overall_rate:
        return "above"
    elif pct < overall_rate:
        return "below"
    return "equal"

py_2_5["vs_overall"] = py_2_5["no_show_pct"].apply(vs_overall_label)
py_2_5 = py_2_5.sort_values(["city", "rank_in_city"]).reset_index(drop=True)

sql_2_5 = pd.read_csv("part-2.5-clinic-ranks.csv").reset_index(drop=True)
print("Row counts match:", len(py_2_5), "vs", len(sql_2_5))
print("py_2_5 cols:", list(py_2_5.columns))
print("sql_2_5 cols:", list(sql_2_5.columns))

# Keep only shared columns in the exact same order and reset index
common_cols_2_5 = [c for c in sql_2_5.columns if c in py_2_5.columns]
py_2_5_aligned = py_2_5[common_cols_2_5].reset_index(drop=True)
sql_2_5_aligned = sql_2_5[common_cols_2_5].reset_index(drop=True)

# Round percentage column identically before comparing
if "no_show_pct" in py_2_5_aligned.columns:
    py_2_5_aligned["no_show_pct"] = pd.to_numeric(py_2_5_aligned["no_show_pct"], errors="coerce").round(2)
    sql_2_5_aligned["no_show_pct"] = pd.to_numeric(sql_2_5_aligned["no_show_pct"], errors="coerce").round(2)

diff_2_5 = py_2_5_aligned.compare(sql_2_5_aligned, result_names=("python", "sql"))
print(diff_2_5 if not diff_2_5.empty else "Part 2.5 matches with zero differences!")

=== TASK 2.2 COMPARISON ===
Row counts match: 6 vs 6
Empty DataFrame
Columns: []
Index: []

=== TASK 2.3 COMPARISON ===
Row counts match: 4 vs 4
Empty DataFrame
Columns: []
Index: []

=== TASK 2.4 COMPARISON ===
py_2_4 columns: ['appt_month', 'appointments', 'no_show_pct', 'rate_diff_vs_prior_month'] shape: (23, 4)
sql_2_4 columns: ['appt_month', 'appointments', 'no_show_pct', 'rate_diff_vs_prior_month'] shape: (6, 4)
Shape mismatch in Part 2.4 — check month filtering or grouping.
Row counts match: 23 vs 6
py_2_4 cols: ['appt_month', 'appointments', 'no_show_pct', 'rate_diff_vs_prior_month'] shape: (23, 4)
sql_2_4 cols: ['appt_month', 'appointments', 'no_show_pct', 'rate_diff_vs_prior_month'] shape: (6, 4)
Shape mismatch — check month rows or filtering in py_2_4 vs sql_2_4.

=== TASK 2.5 COMPARISON ===
Row counts match: 8 vs 8
py_2_5 cols: ['clinic_name', 'city', 'appointments', 'no_shows', 'no_show_pct', 'rank_in_city', 'vs_overall']
sql_2_5 cols: ['clinic_name', 'city', 'appointments

In [ ]:
print("2.2 Without HAVING, two more clinics appear: Gomti Nagar Care Point (Lucknow) has the highest rate, 33.3%, but only 90 appointments; Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%.\nThe pandas counts match the SQL counts exactly.")
print("2.3 (appointments / no-shows / %) are 2.3: 1-7: 390 / 63 / 16.2 · 8-14: 304 / 53 / 17.4 · 15-30: 253 / 76 / 30.0 · 31+: 43 / 26 / 60.5 (appointments / no-shows / %).")
print("2.4 (month · appointments · rate · change), 2.4: 2025-01: 157 · 24.8 · blank; 2025-02: 151 · 20.5 · −4.3; 2025-03: 142 · 20.4 · −0.1; 2025-04: 177 · 20.9 · +0.5; 2025-05: 198 · 23.7 · +2.8; 2025-06: 165 · 21.2 · −2.5")
print("2.5: overall rate is 22.02%. Eight rows: Indore — Vijay Nagar 20.8, rank 1, below · Jaipur — Malviya Nagar 26.8, rank 1, above; Vaishali Nagar 13.8, rank 2, below · Kochi — Edappally 16.2, rank 1, below · Lucknow — Gomti Nagar 33.3, rank 1, above · Nagpur — Sitabuldi 15.9, rank 1, below · Pune — Hadapsar 27.3, rank 1, above; Kothrud 22.5, rank 2, above.")

2.2 Without HAVING, two more clinics appear: Gomti Nagar Care Point (Lucknow) has the highest rate, 33.3%, but only 90 appointments; Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%.
The pandas counts match the SQL counts exactly.
2.3 (appointments / no-shows / %) are 2.3: 1-7: 390 / 63 / 16.2 · 8-14: 304 / 53 / 17.4 · 15-30: 253 / 76 / 30.0 · 31+: 43 / 26 / 60.5 (appointments / no-shows / %).
2.4 (month · appointments · rate · change), 2.4: 2025-01: 157 · 24.8 · blank; 2025-02: 151 · 20.5 · −4.3; 2025-03: 142 · 20.4 · −0.1; 2025-04: 177 · 20.9 · +0.5; 2025-05: 198 · 23.7 · +2.8; 2025-06: 165 · 21.2 · −2.5
2.5: overall rate is 22.02%. Eight rows: Indore — Vijay Nagar 20.8, rank 1, below · Jaipur — Malviya Nagar 26.8, rank 1, above; Vaishali Nagar 13.8, rank 2, below · Kochi — Edappally 16.2, rank 1, below · Lucknow — Gomti Nagar 33.3, rank 1, above · Nagpur — Sitabuldi 15.9, rank 1, below · Pune — Hadapsar 27.3, rank 1, above; Kothrud 22.5, rank 2, above.
